# SQL Patterns — Latest Snapshot per Group

Companion notebook to `eda.ipynb`.

Compares three SQL approaches for extracting the most recent record per instrument
from `positions_history`. All three return identical results on this data set;
the differences matter for correctness under stale data and for performance at scale.

**Sections**
1. Setup
2. Alternative A — `ROW_NUMBER()` subquery (standard SQL)
3. Alternative B — scalar subquery (fragile)
4. Alternative C — correlated subquery

## 1. Setup

In [ ]:
from utils import setup_repo_root

repo_root = setup_repo_root()
print("Working directory:", repo_root)

In [ ]:
import duckdb
import pandas as pd

DB_PATH = "data/reference.duckdb"

with duckdb.connect(DB_PATH, read_only=True) as con:
    history = con.execute(
        "SELECT instrument_id, snapshot_date, market_value_chf, weight "
        "FROM positions_history "
        "ORDER BY instrument_id, snapshot_date"
    ).fetchdf()

print(f"Total rows in positions_history: {len(history)}")
print(f"Distinct instruments:            {history['instrument_id'].nunique()}")
print(f"Distinct snapshot dates:         {sorted(history['snapshot_date'].unique())}")
history.head(12)

## 2. Alternative A — `ROW_NUMBER() OVER (PARTITION BY ...)` as a subquery

This is the standard SQL-compatible form that works on any database engine. It assigns a rank within each instrument group in an inner query, then filters to rank 1 in the outer `WHERE` clause. `QUALIFY` is syntactic sugar that collapses this two-level structure into one.

In [ ]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    rn_subquery_result = con.execute("""
        SELECT instrument_id, snapshot_date
        FROM (
            SELECT
                instrument_id,
                snapshot_date,
                ROW_NUMBER() OVER (
                    PARTITION BY instrument_id
                    ORDER BY snapshot_date DESC
                ) AS rn
            FROM positions_history
        ) ranked
        WHERE rn = 1
        ORDER BY instrument_id
    """).fetchdf()

print(f"Rows returned: {len(rn_subquery_result)}  ✓ (correct)")
print()
print("The two queries below are exactly equivalent — QUALIFY is just shorter:")
print()
print("  -- Standard SQL (subquery required)")
print("  SELECT * FROM (")
print("      SELECT *, ROW_NUMBER() OVER (PARTITION BY instrument_id ORDER BY snapshot_date DESC) AS rn")
print("      FROM positions_history")
print("  ) ranked WHERE rn = 1")
print()
print("  -- DuckDB QUALIFY (no subquery needed)")
print("  SELECT * FROM positions_history")
print("  QUALIFY ROW_NUMBER() OVER (PARTITION BY instrument_id ORDER BY snapshot_date DESC) = 1")
rn_subquery_result

## 3. Alternative B — scalar subquery

This approach assumes every instrument was last updated on the *global* maximum date. If any instrument has a stale or missing snapshot it would be silently dropped, producing a positions table with fewer than 18 rows.

We simulate the failure by temporarily treating one instrument as stale.

In [ ]:
global_max = history["snapshot_date"].max()
stale_mask = history["instrument_id"] == history["instrument_id"].iloc[0]
removed_instrument_id = history["instrument_id"].iloc[0]
simulated = history.copy()
simulated.loc[stale_mask, "snapshot_date"] = global_max - pd.Timedelta(days=7)

scalar_subquery_result = simulated[simulated["snapshot_date"] == global_max]
print(f"Rows returned by scalar-subquery approach: {len(scalar_subquery_result)}")
print(
    f"{removed_instrument_id} (deliberately staled) is missing: "
    f"{removed_instrument_id not in scalar_subquery_result['instrument_id'].values}"
)

## 4. Alternative C — correlated subquery

A correlated subquery scopes the inner `MAX()` to the current instrument, so it handles stale instruments correctly — unlike the scalar subquery in Alternative B.

The cost is performance: the inner `SELECT MAX(...)` re-executes once per row in the outer table, making this **O(N²)** in the number of history rows. The window-function approach is **O(N log N)** because it sorts once and scans once. The difference is negligible for 90 rows but matters at scale.

In [ ]:
with duckdb.connect(DB_PATH, read_only=True) as con:
    correlated_result = con.execute("""
        SELECT instrument_id, snapshot_date
        FROM positions_history p
        WHERE snapshot_date = (
            SELECT MAX(p2.snapshot_date)
            FROM positions_history p2
            WHERE p2.instrument_id = p.instrument_id  -- ← correlated: references outer row
        )
        ORDER BY instrument_id
    """).fetchdf()

print(f"Rows returned: {len(correlated_result)}  ✓ (correct — stale instruments included)")
print()

assert rn_subquery_result.equals(correlated_result), "Results differ!"
print("ROW_NUMBER subquery and correlated subquery return identical results.")
correlated_result